In [12]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
import os
from src.config import Configuration

CONFIG = Configuration()

Configuration initialized
 - seed: 42
 - masked_train: False
 - masked_test: False
 - dataset: brats
 - epochs: 30 (SSL), 20 (Survival)
 - use_radiomics: False
 - dynamic_dropout: False
 - batch_size: 32 (SSL), 16 (Survival)


In [14]:
CHECKPOINTS = [
    ("../models/brats-All_masks-No_Radiomics_2026-05-25-15-59-58/survival_checkpoint_best.ckpt",False, 'brats-All_masks-No_Radiomics'),
    ("../models/brats-All_masks-Radiomics_2026-05-25-16-01-19/survival_checkpoint_best.ckpt",True, 'brats-All_masks-Radiomics'),
    ("../models/brats-No_masks-No_Radiomics_2026-05-25-16-02-34/survival_checkpoint_best.ckpt",False, 'brats-No_masks-No_Radiomics'),
    ("../models/brats-No_masks-Radiomics_2026-05-25-16-04-14/survival_checkpoint_best.ckpt",True, 'brats-No_masks-Radiomics'),
    # ("../models/brats-Train_masks-No_Radiomics_2026-05-25-16-05-50/survival_checkpoint_best.ckpt",False, 'brats-Train_masks-No_Radiomics'),
    # ("../models/brats-Train_masks-Radiomics_2026-05-25-16-07-14/survival_checkpoint_best.ckpt",True, 'brats-Train_masks-Radiomics'),
    # ("../models/brats-No_masks-No_Radiomics-test_2026-05-25-16-25-48/survival_checkpoint_best.ckpt",False, 'brats-No_masks-No_Radiomics-test'),
    # ("../models/brats-No_masks-Radiomics-test_2026-05-25-16-27-10/survival_checkpoint_best.ckpt",True, 'brats-No_masks-Radiomics-test'),
    ("../models/Upenn-All_masks-No_Radiomics_2026-05-25-16-08-40/survival_checkpoint_best.ckpt",False, 'Upenn-All_masks-No_Radiomics'),
    ("../models/Upenn-All_masks-Radiomics_2026-05-25-16-11-40/survival_checkpoint_best.ckpt",True, 'Upenn-All_masks-Radiomics'),
    ("../models/Upenn-No_masks-No_Radiomics_2026-05-25-16-13-00/survival_checkpoint_best.ckpt",False, 'Upenn-No_masks-No_Radiomics'),
    ("../models/Upenn-No_masks-Radiomics_2026-05-25-16-14-39/survival_checkpoint_best.ckpt",True, 'Upenn-No_masks-Radiomics'),
    # ("../models/Upenn-Train_masks-No_Radiomics_2026-05-25-16-16-13/survival_checkpoint_best.ckpt",False, 'Upenn-Train_masks-No_Radiomics'),
    # ("../models/Upenn-Train_masks-Radiomics_2026-05-25-16-17-37/survival_checkpoint_best.ckpt",True, 'Upenn-Train_masks-Radiomics'),
    # ("../models/Upenn-No_masks-No_Radiomics-test_2026-05-25-16-28-33/survival_checkpoint_best.ckpt",False, 'Upenn-No_masks-No_Radiomics-test'),
    # ("../models/Upenn-No_masks-Radiomics-test_2026-05-25-16-31-15/survival_checkpoint_best.ckpt",True, 'Upenn-No_masks-Radiomics-test'),
]

good_models = [
    "brats-All_masks-No_Radiomics",
    'brats-All_masks-Radiomics',

    "Upenn-All_masks-No_Radiomics",
    'Upenn-All_masks-Radiomics',

    "brats-No_masks-No_Radiomics",
    "Upenn-No_masks-No_Radiomics",
]

for path, _, label in CHECKPOINTS:
    if not os.path.exists(path):
        print(f"MISSING: {path}")
    else:
        print(f"  OK: {path}")

  OK: ../models/brats-All_masks-No_Radiomics_2026-05-25-15-59-58/survival_checkpoint_best.ckpt
  OK: ../models/brats-All_masks-Radiomics_2026-05-25-16-01-19/survival_checkpoint_best.ckpt
  OK: ../models/brats-No_masks-No_Radiomics_2026-05-25-16-02-34/survival_checkpoint_best.ckpt
  OK: ../models/brats-No_masks-Radiomics_2026-05-25-16-04-14/survival_checkpoint_best.ckpt
  OK: ../models/Upenn-All_masks-No_Radiomics_2026-05-25-16-08-40/survival_checkpoint_best.ckpt
  OK: ../models/Upenn-All_masks-Radiomics_2026-05-25-16-11-40/survival_checkpoint_best.ckpt
  OK: ../models/Upenn-No_masks-No_Radiomics_2026-05-25-16-13-00/survival_checkpoint_best.ckpt
  OK: ../models/Upenn-No_masks-Radiomics_2026-05-25-16-14-39/survival_checkpoint_best.ckpt


In [15]:
import copy
import os

import torch
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from tqdm.notebook import tqdm

from src.data.UPennGBMDataset import UPennGBMDataset
from src.models.survival_predictor import MultimodalSurvivalPredictor
from maikol_utils.print_utils import print_color

REGIONS = ["ED", "ET", "NC"]
CHANNELS = ["canal_T1", "canal_T1GD", "canal_T2", "canal_FLAIR"]
RADIOMIC_REGIONS = ["ED", "ET", "NC"]
TABULAR_FEATURES = ["Gender", "Age", "KPS", "IDH1", "MGMT", "GTR"]

IMAGE_CHANNEL_IDX = {"canal_T1": 0, "canal_T1GD": 1, "canal_T2": 2, "canal_FLAIR": 3}

TABULAR_COL_INDICES = {
    "Gender": [0],
    "Age": [1],
    "KPS": [2, 3, 4],
    "IDH1": [5, 6, 7],
    "MGMT": [8, 9, 10],
    "GTR": [11, 12, 13],
}

BASE_TEMPLATE = {
    "canal_T1": True, "canal_T1GD": True, "canal_T2": True, "canal_FLAIR": True,
    "RADIOMIC": {"ET": True, "ED": True, "NC": True},
    "TABULAR": {"Gender": True, "Age": True, "KPS": True,
                 "IDH1": True, "MGMT": True, "GTR": True},
}

In [16]:
def generate_test_configs():
    configs = {}
    configs["base"] = copy.deepcopy(BASE_TEMPLATE)

    for ch in CHANNELS:
        cfg = copy.deepcopy(BASE_TEMPLATE)
        cfg[ch] = False
        configs[f"no_{ch}"] = cfg

    for region in RADIOMIC_REGIONS:
        cfg = copy.deepcopy(BASE_TEMPLATE)
        cfg["RADIOMIC"][region] = False
        configs[f"no_radiomic_{region}"] = cfg

    for feat in TABULAR_FEATURES:
        cfg = copy.deepcopy(BASE_TEMPLATE)
        cfg["TABULAR"][feat] = False
        configs[f"no_tabular_{feat}"] = cfg

    cfg = copy.deepcopy(BASE_TEMPLATE)
    for ch in CHANNELS:
        cfg[ch] = False
    configs["no_all_channels"] = cfg

    cfg = copy.deepcopy(BASE_TEMPLATE)
    for region in RADIOMIC_REGIONS:
        cfg["RADIOMIC"][region] = False
    configs["no_all_radiomic"] = cfg

    cfg = copy.deepcopy(BASE_TEMPLATE)
    for feat in TABULAR_FEATURES:
        cfg["TABULAR"][feat] = False
    configs["no_all_tabular"] = cfg

    return configs


def apply_config_to_batch(batch, config, device):
    image = batch['image'].to(device).clone()          # ← .clone()
    tabular = batch['tabular'].to(device).clone()      # ← .clone()
    labels = batch['label'].to(device)

    image_mask = batch['image_mask'].to(device).clone()  # ← .clone()
    for ch_name, ch_idx in IMAGE_CHANNEL_IDX.items():
        if not config[ch_name]:
            image[:, ch_idx] = 0.0
            image_mask[:, ch_idx] = 0.0

    radiomic = {
        'ED': batch['radiomic_ED'].to(device).clone(),   # ← .clone()
        'ET': batch['radiomic_ET'].to(device).clone(),
        'NC': batch['radiomic_NC'].to(device).clone(),
    }
    radiomic_mask = {}
    for region in REGIONS:
        natural = batch[f'radiomic_mask_{region}'].to(device).clone()  # ← .clone()
        if not config["RADIOMIC"][region]:
            radiomic[region] = torch.zeros_like(radiomic[region])      # ← zeroing values
            natural = torch.zeros_like(natural)
        radiomic_mask[region] = natural

    tabular_mask = batch['tabular_mask'].to(device).clone()  # ← .clone()
    for feat in TABULAR_FEATURES:
        if not config["TABULAR"][feat]:
            for col_idx in TABULAR_COL_INDICES[feat]:
                tabular[:, col_idx] = 0.0
                tabular_mask[:, col_idx] = 0.0

    return image, tabular, labels, radiomic, radiomic_mask, tabular_mask, image_mask

In [17]:
def run_ablation(model, test_loader, device):
    model.eval()
    configs = generate_test_configs()
    configs_do = ["base", "no_all_channels", "no_all_radiomic", "no_all_tabular"]
    configs = {k: v for k, v in configs.items() if k in configs_do}
    results = {}

    with torch.no_grad():
        for cfg_name, cfg in tqdm(configs.items(), desc="Configs"):
            all_preds, all_labels = [], []

            for batch in tqdm(test_loader, desc=f"  [{cfg_name}]", leave=False):
                image, tabular, labels, radiomic, radiomic_mask, tabular_mask, image_mask = \
                    apply_config_to_batch(batch, cfg, device)

                preds = model(image, tabular, radiomic, radiomic_mask,
                              tabular_mask=tabular_mask,
                              image_mask=image_mask)
                all_preds.extend(torch.argmax(preds, dim=1).cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

            results[cfg_name] = {
                'f1': float(f1_score(all_labels, all_preds, average='weighted')),
                'precision': float(precision_score(all_labels, all_preds, average='weighted', zero_division=0)),
                'recall': float(recall_score(all_labels, all_preds, average='weighted', zero_division=0)),
                'accuracy': float(accuracy_score(all_labels, all_preds)),
            }

    return results

In [18]:
def load_model_for_ablation(checkpoint_path, use_radiomics, CONFIG):
    model = MultimodalSurvivalPredictor(
        num_classes=len(CONFIG.labels),
        embed_dim=CONFIG.ssl_embed_dim,
        num_heads=CONFIG.ssl_num_heads,
        dropout=CONFIG.ssl_dropout,
        in_channels=4,
        patch_size=CONFIG.ssl_patch_size,
        vit_depth=CONFIG.ssl_vit_depth,
        vol_size=CONFIG.ssl_vol_size,
        pos_embed=CONFIG.pos_embed,
        use_radiomics=use_radiomics,
    )

    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    state_dict = checkpoint.get("model_state_dict", checkpoint.get("state_dict", checkpoint))
    # Lightning checkpoints prefix keys with 'model.' — strip it for the bare predictor
    state_dict = {k.removeprefix('model.'): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=True)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"  {use_radiomics=}, params={n_params:,}")

    return model

In [19]:
from maikol_utils.file_utils import load_json, save_json 
from src.data import SurvivalInferencePipeline

all_results = load_json('../models/results.json')
if all_results:
    print_color("Loaded existing results from '../models/results.json'", 'green')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    

    all_results = {}

    for ckpt_path, use_radiomics, label in CHECKPOINTS:
        test_dataset = UPennGBMDataset(
            CONFIG=CONFIG,
            partition='test',
            apply_mask=False,
            transform=SurvivalInferencePipeline()
        )
        test_loader = DataLoader(
            test_dataset,
            batch_size=CONFIG.survival_batch_size,
            shuffle=False,
            num_workers=CONFIG.survival_num_workers,
        )  # Just to trigger any potential loading issues early
        print_color(f"\n{'='*60}", 'cyan')
        print_color(f"  {label}", 'cyan')
        print_color(f"  {ckpt_path}", 'cyan')
        print_color(f"{'='*60}", 'cyan')

        model = load_model_for_ablation(ckpt_path, use_radiomics, CONFIG)
        model = model.to(device)

        results = run_ablation(model, test_loader, device)
        all_results[label] = results

        del model
        torch.cuda.empty_cache()
    save_json('../models/results.json', all_results)

Loading output from ../models/results.json...
Loaded existing results from '../models/results.json'


In [20]:
for label, results in all_results.items():
    print()
    print_color(f"  {label}", 'cyan')
    print("-" * 80)
    header = f"  {'Config':<28} {'F1':>8} {'Precision':>10} {'Recall':>8} {'Accuracy':>10}"
    print(header)
    print("  " + "-" * 76)
    for cfg_name, res in results.items():
        marker = "  <-- baseline" if cfg_name == "base" else ""
        print(f"  {cfg_name:<28} {res['f1']:>8.4f} {res['precision']:>10.4f} {res['recall']:>8.4f} {res['accuracy']:>10.4f} {marker}")
    print()


  brats-All_masks-No_Radiomics
--------------------------------------------------------------------------------
  Config                             F1  Precision   Recall   Accuracy
  ----------------------------------------------------------------------------
  base                           0.6261     0.6672   0.6393     0.6393   <-- baseline
  no_all_channels                0.6065     0.6530   0.6230     0.6230 
  no_all_radiomic                0.6261     0.6672   0.6393     0.6393 
  no_all_tabular                 0.3243     0.2419   0.4918     0.4918 


  brats-All_masks-Radiomics
--------------------------------------------------------------------------------
  Config                             F1  Precision   Recall   Accuracy
  ----------------------------------------------------------------------------
  base                           0.6006     0.6162   0.6066     0.6066   <-- baseline
  no_all_channels                0.5895     0.5917   0.5902     0.5902 
  no_all_radiomi

In [24]:
import pandas as pd
import plotly.express as px

rows = []

for model_name, results in all_results.items():
    for cfg_name, metrics in results.items():

        rows.append({
            "model": model_name,
            "config": cfg_name,
            "f1": metrics["f1"],
        })
good_models = [
    "brats-All_masks-No_Radiomics",
    'brats-All_masks-Radiomics',

    "Upenn-All_masks-No_Radiomics",
    'Upenn-All_masks-Radiomics',

    "brats-No_masks-Radiomics",
    "Upenn-No_masks-Radiomics",
    
    "brats-No_masks-No_Radiomics",
    "Upenn-No_masks-No_Radiomics",
]
rows = [
    row 
    for row in rows
    if row['model'] in good_models
]

df_plot = pd.DataFrame(rows)

config_order = [
    "base",
    # "no_canal_T1", "no_canal_T1GD", "no_canal_T2", "no_canal_FLAIR",
    # "no_radiomic_ED", "no_radiomic_ET", "no_radiomic_NC",
    # "no_tabular_Gender", "no_tabular_Age", "no_tabular_KPS",
    # "no_tabular_IDH1", "no_tabular_MGMT", "no_tabular_GTR",
    "no_all_channels", "no_all_radiomic", "no_all_tabular",
]

df_plot = df_plot[df_plot['config'].isin(config_order)]


fig = px.scatter(
    df_plot,
    x="config",
    y="f1",
    color="model",
    category_orders={"config": config_order},
)

fig.update_traces(marker=dict(size=10))

fig.update_layout(
    title="F1 score by ablation config",
    xaxis_title="Ablation Config",
    yaxis_title="F1",
    height=700,
)

fig.show()

In [25]:
df_plot

,model,config,f1
0,brats-All_masks-No_Radiomics,base,0.626120
1,brats-All_masks-No_Radiomics,no_all_channels,0.606507
2,brats-All_masks-No_Radiomics,no_all_radiomic,0.626120
3,brats-All_masks-No_Radiomics,no_all_tabular,0.324266
4,brats-All_masks-Radiomics,base,0.600557
5,brats-All_masks-Radiomics,no_all_channels,0.589502
6,brats-All_masks-Radiomics,no_all_radiomic,0.665646
7,brats-All_masks-Radiomics,no_all_tabular,0.573082
8,brats-No_masks-No_Radiomics,base,0.615527
9,brats-No_masks-No_Radiomics,no_all_channels,0.638178


In [26]:
a = ["BraTS2021_00367", "BraTS2021_01208", "BraTS2021_01537", "BraTS2021_00625", "BraTS2021_00166", "BraTS2021_00116", "BraTS2021_00017", "BraTS2021_00301", "BraTS2021_01407", "BraTS2021_00234", "BraTS2021_01376", "BraTS2021_01216", "BraTS2021_00526", "BraTS2021_01049", "BraTS2021_01557", "BraTS2021_01452", "BraTS2021_00654", "BraTS2021_00346", "BraTS2021_01125", "BraTS2021_01428", "BraTS2021_00773", "BraTS2021_01260", "BraTS2021_01314", "BraTS2021_00689", "BraTS2021_00210", "BraTS2021_00604", "BraTS2021_01273", "BraTS2021_00404", "BraTS2021_01113", "BraTS2021_01158", "BraTS2021_00565", "BraTS2021_00123", "BraTS2021_00506", "BraTS2021_01059", "BraTS2021_00024", "BraTS2021_00052", "BraTS2021_01454", "BraTS2021_00217", "BraTS2021_00299", "BraTS2021_01084", "BraTS2021_00500", "BraTS2021_01430", "BraTS2021_01414", "BraTS2021_01392", "BraTS2021_01053", "BraTS2021_01263", "BraTS2021_01402", "BraTS2021_01373", "BraTS2021_00836", "BraTS2021_01133", "BraTS2021_00683", "BraTS2021_01182", "BraTS2021_00552", "BraTS2021_00619", "BraTS2021_01258", "BraTS2021_00386", "BraTS2021_00088", "BraTS2021_01122", "BraTS2021_01544", "BraTS2021_01071", "BraTS2021_01274", "BraTS2021_01617", "BraTS2021_01044", "BraTS2021_01244", "BraTS2021_00649", "BraTS2021_01011", "BraTS2021_00391", "BraTS2021_00571", "BraTS2021_00540", "BraTS2021_01581", "BraTS2021_00598", "BraTS2021_01132", "BraTS2021_00154", "BraTS2021_01034", "BraTS2021_01596", "BraTS2021_00324", "BraTS2021_00102", "BraTS2021_00520", "BraTS2021_01297", "BraTS2021_00594", "BraTS2021_00602", "BraTS2021_00228", "BraTS2021_01530", "BraTS2021_01465", "BraTS2021_01296", "BraTS2021_00568", "BraTS2021_01575", "BraTS2021_00071", "BraTS2021_00814", "BraTS2021_01241", "BraTS2021_00423", "BraTS2021_01528", "BraTS2021_01419", "BraTS2021_01126", "BraTS2021_01022", "BraTS2021_01489", "BraTS2021_00380", "BraTS2021_01439", "BraTS2021_01201", "BraTS2021_01000", "BraTS2021_00797", "BraTS2021_01243", "BraTS2021_00746", "BraTS2021_00031", "BraTS2021_00999", "BraTS2021_01261", "BraTS2021_01563", "BraTS2021_01118", "BraTS2021_00656", "BraTS2021_01333", "BraTS2021_01548", "BraTS2021_00107", "BraTS2021_00176", "BraTS2021_00250", "BraTS2021_01238", "BraTS2021_00574", "BraTS2021_01234", "BraTS2021_01583", "BraTS2021_01569", "BraTS2021_01573", "BraTS2021_01626", "BraTS2021_01262", "BraTS2021_01542", "BraTS2021_01412", "BraTS2021_01482", "BraTS2021_00532", "BraTS2021_00750", "BraTS2021_00684", "BraTS2021_01616", "BraTS2021_01055", "BraTS2021_01494", "BraTS2021_00130", "BraTS2021_01566", "BraTS2021_01039", "BraTS2021_01310", "BraTS2021_00126", "BraTS2021_01286", "BraTS2021_00687", "BraTS2021_00016", "BraTS2021_00767", "BraTS2021_00068", "BraTS2021_01639", "BraTS2021_00733", "BraTS2021_00686", "BraTS2021_00730", "BraTS2021_00784", "BraTS2021_00445", "BraTS2021_00397", "BraTS2021_01365", "BraTS2021_00360", "BraTS2021_00195", "BraTS2021_01607", "BraTS2021_01223", "BraTS2021_00336", "BraTS2021_01023", "BraTS2021_01540", "BraTS2021_00421", "BraTS2021_00221", "BraTS2021_00120", "BraTS2021_00453", "BraTS2021_00417", "BraTS2021_00106", "BraTS2021_01067", "BraTS2021_01341", "BraTS2021_00472", "BraTS2021_01202", "BraTS2021_01290", "BraTS2021_01130", "BraTS2021_00140", "BraTS2021_01001", "BraTS2021_00162", "BraTS2021_01255", "BraTS2021_01662", "BraTS2021_00819", "BraTS2021_00209", "BraTS2021_00576", "BraTS2021_00022", "BraTS2021_00622", "BraTS2021_01082", "BraTS2021_00430", "BraTS2021_00468", "BraTS2021_01352", "BraTS2021_01406", "BraTS2021_01527", "BraTS2021_00058", "BraTS2021_00400", "BraTS2021_01624", "BraTS2021_01368", "BraTS2021_00115", "BraTS2021_01190", "BraTS2021_01367", "BraTS2021_00650", "BraTS2021_01303", "BraTS2021_00074", "BraTS2021_01336", "BraTS2021_00046", "BraTS2021_01308", "BraTS2021_01425", "BraTS2021_01106", "BraTS2021_01031", "BraTS2021_00554", "BraTS2021_01460", "BraTS2021_00317", "BraTS2021_00676", "BraTS2021_01270", "BraTS2021_01401", "BraTS2021_00157", "BraTS2021_01146", "BraTS2021_00551", "BraTS2021_00441", "BraTS2021_00300", "BraTS2021_01196", "BraTS2021_01420", "BraTS2021_01350", "BraTS2021_01179", "BraTS2021_00219", "BraTS2021_01294", "BraTS2021_00110", "BraTS2021_00751", "BraTS2021_01329", "BraTS2021_00455", "BraTS2021_00416", "BraTS2021_00056", "BraTS2021_01278", "BraTS2021_00291", "BraTS2021_00051", "BraTS2021_01380", "BraTS2021_01279", "BraTS2021_01503", "BraTS2021_01074", "BraTS2021_01171", "BraTS2021_00642", "BraTS2021_01007", "BraTS2021_00344", "BraTS2021_01495", "BraTS2021_01477", "BraTS2021_01124", "BraTS2021_01019", "BraTS2021_01637", "BraTS2021_01579", "BraTS2021_01638", "BraTS2021_01246", "BraTS2021_01326", "BraTS2021_00211", "BraTS2021_01028", "BraTS2021_00828", "BraTS2021_01618", "BraTS2021_00196", "BraTS2021_01479", "BraTS2021_01222", "BraTS2021_00718", "BraTS2021_00011", "BraTS2021_01568", "BraTS2021_00128", "BraTS2021_00261", "BraTS2021_00158", "BraTS2021_01587", "BraTS2021_00138", "BraTS2021_01404", "BraTS2021_00204", "BraTS2021_01115", "BraTS2021_01317", "BraTS2021_00425", "BraTS2021_00530", "BraTS2021_01562", "BraTS2021_00693", "BraTS2021_01254", "BraTS2021_00760", "BraTS2021_00572", "BraTS2021_01447", "BraTS2021_01534", "BraTS2021_00557", "BraTS2021_00388", "BraTS2021_00545", "BraTS2021_01449", "BraTS2021_01319", "BraTS2021_00137", "BraTS2021_00529", "BraTS2021_01654", "BraTS2021_01172", "BraTS2021_00178", "BraTS2021_01160", "BraTS2021_00705", "BraTS2021_00325", "BraTS2021_01066", "BraTS2021_01075", "BraTS2021_00838", "BraTS2021_01131", "BraTS2021_00547", "BraTS2021_01531", "BraTS2021_00249", "BraTS2021_01232", "BraTS2021_01604", "BraTS2021_01230", "BraTS2021_00267", "BraTS2021_00418", "BraTS2021_01189", "BraTS2021_00782", "BraTS2021_00675", "BraTS2021_00510", "BraTS2021_00293", "BraTS2021_00155", "BraTS2021_01135", "BraTS2021_01094", "BraTS2021_01309", "BraTS2021_01346", "BraTS2021_01121", "BraTS2021_00351", "BraTS2021_00523", "BraTS2021_01525", "BraTS2021_00341", "BraTS2021_01178", "BraTS2021_00623", "BraTS2021_01057", "BraTS2021_01325", "BraTS2021_01403", "BraTS2021_01383", "BraTS2021_00528", "BraTS2021_01591", "BraTS2021_01487", "BraTS2021_00136", "BraTS2021_00375", "BraTS2021_01521", "BraTS2021_00804", "BraTS2021_01601", "BraTS2021_01061", "BraTS2021_01195", "BraTS2021_01253", "BraTS2021_01516", "BraTS2021_00203", "BraTS2021_00246", "BraTS2021_00186", "BraTS2021_01200", "BraTS2021_01228", "BraTS2021_01443", "BraTS2021_01081", "BraTS2021_01612", "BraTS2021_01229", "BraTS2021_01077", "BraTS2021_00788", "BraTS2021_01484", "BraTS2021_00242", "BraTS2021_01394", "BraTS2021_00494", "BraTS2021_01206", "BraTS2021_01320", "BraTS2021_00332", "BraTS2021_00757", "BraTS2021_01069", "BraTS2021_01145", "BraTS2021_01485", "BraTS2021_01613", "BraTS2021_00064", "BraTS2021_00837", "BraTS2021_01444", "BraTS2021_00172", "BraTS2021_01058", "BraTS2021_01268", "BraTS2021_01302", "BraTS2021_01398", "BraTS2021_00538", "BraTS2021_01227", "BraTS2021_01535", "BraTS2021_00334", "BraTS2021_01492", "BraTS2021_00567", "BraTS2021_00389", "BraTS2021_00667", "BraTS2021_01369", "BraTS2021_01247", "BraTS2021_01104", "BraTS2021_01602", "BraTS2021_00704", "BraTS2021_01186", "BraTS2021_01446", "BraTS2021_00127", "BraTS2021_01397", "BraTS2021_00112", "BraTS2021_01582", "BraTS2021_00329", "BraTS2021_01029", "BraTS2021_01056", "BraTS2021_00459", "BraTS2021_00262", "BraTS2021_00697", "BraTS2021_00311", "BraTS2021_00149", "BraTS2021_01594", "BraTS2021_01021", "BraTS2021_00018", "BraTS2021_00502", "BraTS2021_01331", "BraTS2021_00544", "BraTS2021_01235", "BraTS2021_00511", "BraTS2021_01248", "BraTS2021_00170", "BraTS2021_01166", "BraTS2021_00144", "BraTS2021_00348", "BraTS2021_01259", "BraTS2021_01097", "BraTS2021_00578", "BraTS2021_00550", "BraTS2021_00452", "BraTS2021_00036", "BraTS2021_01236", "BraTS2021_01634", "BraTS2021_00237", "BraTS2021_00777", "BraTS2021_00241", "BraTS2021_00542", "BraTS2021_01508", "BraTS2021_01117", "BraTS2021_01453", "BraTS2021_00159", "BraTS2021_01620", "BraTS2021_00183", "BraTS2021_01047", "BraTS2021_01032", "BraTS2021_01018", "BraTS2021_01209", "BraTS2021_01497", "BraTS2021_00048", "BraTS2021_00061", "BraTS2021_01324", "BraTS2021_01518", "BraTS2021_01207", "BraTS2021_00081", "BraTS2021_01590", "BraTS2021_00026", "BraTS2021_01211", "BraTS2021_01360", "BraTS2021_00366", "BraTS2021_00608", "BraTS2021_00402", "BraTS2021_01139", "BraTS2021_00498", "BraTS2021_01600", "BraTS2021_01510", "BraTS2021_00304", "BraTS2021_00480", "BraTS2021_01215", "BraTS2021_00377", "BraTS2021_01606", "BraTS2021_01558", "BraTS2021_00674", "BraTS2021_01480", "BraTS2021_01472", "BraTS2021_00121", "BraTS2021_00831", "BraTS2021_00575", "BraTS2021_00043", "BraTS2021_01543", "BraTS2021_01264", "BraTS2021_00682", "BraTS2021_00239", "BraTS2021_00587", "BraTS2021_00496", "BraTS2021_01020", "BraTS2021_00045", "BraTS2021_01559", "BraTS2021_01165", "BraTS2021_00395", "BraTS2021_01343", "BraTS2021_00709", "BraTS2021_00316", "BraTS2021_00803", "BraTS2021_01177", "BraTS2021_01359", "BraTS2021_01524", "BraTS2021_00431", "BraTS2021_00517", "BraTS2021_01536", "BraTS2021_01386", "BraTS2021_01650", "BraTS2021_01161", "BraTS2021_00028", "BraTS2021_01387", "BraTS2021_00569", "BraTS2021_01315", "BraTS2021_00298", "BraTS2021_00097", "BraTS2021_00601", "BraTS2021_00680", "BraTS2021_01393", "BraTS2021_01153", "BraTS2021_00590", "BraTS2021_01088", "BraTS2021_01584", "BraTS2021_00309", "BraTS2021_00220", "BraTS2021_01451", "BraTS2021_00331", "BraTS2021_01442", "BraTS2021_01595", "BraTS2021_01378", "BraTS2021_01437", "BraTS2021_01554", "BraTS2021_01517", "BraTS2021_01295", "BraTS2021_01100", "BraTS2021_01473", "BraTS2021_00113", "BraTS2021_00636", "BraTS2021_00399", "BraTS2021_01348", "BraTS2021_00706", "BraTS2021_00765", "BraTS2021_01276", "BraTS2021_00353", "BraTS2021_00376", "BraTS2021_01490", "BraTS2021_01478", "BraTS2021_00231", "BraTS2021_00645", "BraTS2021_01127", "BraTS2021_00615", "BraTS2021_00736", "BraTS2021_01043", "BraTS2021_00640", "BraTS2021_00457", "BraTS2021_01549", "BraTS2021_00820", "BraTS2021_01382", "BraTS2021_01073", "BraTS2021_00192", "BraTS2021_00659", "BraTS2021_01456", "BraTS2021_01529", "BraTS2021_01445", "BraTS2021_01388", "BraTS2021_00677", "BraTS2021_00759", "BraTS2021_00254", "BraTS2021_01589", "BraTS2021_01576", "BraTS2021_01299", "BraTS2021_00008", "BraTS2021_01251", "BraTS2021_00349", "BraTS2021_01192", "BraTS2021_00725", "BraTS2021_01614", "BraTS2021_00284", "BraTS2021_01004", "BraTS2021_01578", "BraTS2021_00599", "BraTS2021_01611", "BraTS2021_01338", "BraTS2021_01041", "BraTS2021_00690", "BraTS2021_01322", "BraTS2021_00070", "BraTS2021_01574", "BraTS2021_00212", "BraTS2021_00289", "BraTS2021_00322", "BraTS2021_00449", "BraTS2021_01370", "BraTS2021_01433", "BraTS2021_00495", "BraTS2021_01361", "BraTS2021_01064", "BraTS2021_01291", "BraTS2021_00756", "BraTS2021_00409", "BraTS2021_00122", "BraTS2021_01090", "BraTS2021_00243", "BraTS2021_00688", "BraTS2021_01173", "BraTS2021_01330", "BraTS2021_01560", "BraTS2021_01116", "BraTS2021_01266", "BraTS2021_01608", "BraTS2021_00469", "BraTS2021_01086", "BraTS2021_01037", "BraTS2021_01395", "BraTS2021_01250", "BraTS2021_01312", "BraTS2021_00413", "BraTS2021_00442", "BraTS2021_00000", "BraTS2021_00805", "BraTS2021_00588", "BraTS2021_00436", "BraTS2021_00839", "BraTS2021_01046", "BraTS2021_01658", "BraTS2021_01109", "BraTS2021_01418", "BraTS2021_01154", "BraTS2021_00607", "BraTS2021_00281", "BraTS2021_01467", "BraTS2021_01539", "BraTS2021_00426", "BraTS2021_01434", "BraTS2021_00584", "BraTS2021_01593", "BraTS2021_00591", "BraTS2021_01239", "BraTS2021_00448", "BraTS2021_00740", "BraTS2021_01342", "BraTS2021_00177", "BraTS2021_01304", "BraTS2021_01371", "BraTS2021_01498", "BraTS2021_01504", "BraTS2021_01096", "BraTS2021_01550", "BraTS2021_00621", "BraTS2021_00533", "BraTS2021_01457", "BraTS2021_01102", "BraTS2021_01152", "BraTS2021_01288", "BraTS2021_00793", "BraTS2021_00764", "BraTS2021_01522", "BraTS2021_01267", "BraTS2021_01462", "BraTS2021_01644", "BraTS2021_00369", "BraTS2021_01399", "BraTS2021_01005", "BraTS2021_00563", "BraTS2021_01431", "BraTS2021_00747", "BraTS2021_01036", "BraTS2021_00626", "BraTS2021_01349", "BraTS2021_01162", "BraTS2021_00507", "BraTS2021_00624", "BraTS2021_01026", "BraTS2021_00152", "BraTS2021_01212", "BraTS2021_01328", "BraTS2021_01010", "BraTS2021_00259", "BraTS2021_01353", "BraTS2021_01300", "BraTS2021_00044", "BraTS2021_00488", "BraTS2021_00791", "BraTS2021_01111", "BraTS2021_00581", "BraTS2021_00744", "BraTS2021_01072", "BraTS2021_00516", "BraTS2021_01198", "BraTS2021_00105", "BraTS2021_01570", "BraTS2021_00481", "BraTS2021_00580", "BraTS2021_00282", "BraTS2021_01091", "BraTS2021_00033", "BraTS2021_01541", "BraTS2021_00327", "BraTS2021_00443", "BraTS2021_00514", "BraTS2021_01486", "BraTS2021_00290", "BraTS2021_00318", "BraTS2021_00685", "BraTS2021_01567", "BraTS2021_00558", "BraTS2021_01458", "BraTS2021_01134", "BraTS2021_01609", "BraTS2021_00387", "BraTS2021_00356", "BraTS2021_01168", "BraTS2021_01351", "BraTS2021_00596", "BraTS2021_01217", "BraTS2021_00491", "BraTS2021_01170", "BraTS2021_00260", "BraTS2021_00035", "BraTS2021_00597", "BraTS2021_00258", "BraTS2021_00236", "BraTS2021_01429", "BraTS2021_01400", "BraTS2021_01183", "BraTS2021_01051", "BraTS2021_01427", "BraTS2021_00823", "BraTS2021_01214", "BraTS2021_00519", "BraTS2021_00146", "BraTS2021_01098", "BraTS2021_01621", "BraTS2021_01114", "BraTS2021_01292", "BraTS2021_00525", "BraTS2021_01592", "BraTS2021_01231", "BraTS2021_00537", "BraTS2021_00446", "BraTS2021_00062", "BraTS2021_01552", "BraTS2021_01633", "BraTS2021_00214", "BraTS2021_00809", "BraTS2021_01012", "BraTS2021_01321", "BraTS2021_00054", "BraTS2021_00802", "BraTS2021_00216", "BraTS2021_01123", "BraTS2021_00111", "BraTS2021_01586", "BraTS2021_01337", "BraTS2021_00824", "BraTS2021_00616", "BraTS2021_01138", "BraTS2021_00808", "BraTS2021_00639", "BraTS2021_01175", "BraTS2021_00235", "BraTS2021_00379", "BraTS2021_01068", "BraTS2021_00187", "BraTS2021_01205", "BraTS2021_00493", "BraTS2021_00005", "BraTS2021_00066", "BraTS2021_01561", "BraTS2021_01265", "BraTS2021_01409", "BraTS2021_01377", "BraTS2021_00132", "BraTS2021_00078", "BraTS2021_01501", "BraTS2021_01507", "BraTS2021_00714", "BraTS2021_00737", "BraTS2021_01423", "BraTS2021_01065", "BraTS2021_01119", "BraTS2021_00294", "BraTS2021_00505", "BraTS2021_01150", "BraTS2021_00303", "BraTS2021_01035", "BraTS2021_00406", "BraTS2021_00392", "BraTS2021_00227", "BraTS2021_00810", "BraTS2021_01176", "BraTS2021_00150", "BraTS2021_01641", "BraTS2021_01471", "BraTS2021_00193", "BraTS2021_00147", "BraTS2021_01095", "BraTS2021_00432", "BraTS2021_00811", "BraTS2021_01252", "BraTS2021_01099", "BraTS2021_00012", "BraTS2021_01657", "BraTS2021_01085", "BraTS2021_01242", "BraTS2021_01042", "BraTS2021_00440", "BraTS2021_01136", "BraTS2021_00800", "BraTS2021_01269", "BraTS2021_00630", "BraTS2021_01381", "BraTS2021_00410", "BraTS2021_00543", "BraTS2021_00350", "BraTS2021_01316", "BraTS2021_01375", "BraTS2021_01318", "BraTS2021_00513", "BraTS2021_01580", "BraTS2021_01347", "BraTS2021_00739", "BraTS2021_01424", "BraTS2021_00792", "BraTS2021_00109", "BraTS2021_00142", "BraTS2021_00830", "BraTS2021_01052", "BraTS2021_01585", "BraTS2021_00014", "BraTS2021_00108", "BraTS2021_01185", "BraTS2021_01354", "BraTS2021_01481", "BraTS2021_00433", "BraTS2021_01076", "BraTS2021_01103", "BraTS2021_00188", "BraTS2021_00577", "BraTS2021_01415", "BraTS2021_00343", "BraTS2021_01390", "BraTS2021_01149", "BraTS2021_01101", "BraTS2021_01631", "BraTS2021_00378", "BraTS2021_01538", "BraTS2021_00768", "BraTS2021_00834", "BraTS2021_01013", "BraTS2021_01089", "BraTS2021_01599", "BraTS2021_00118", "BraTS2021_01050", "BraTS2021_00060", "BraTS2021_01024", "BraTS2021_01610", "BraTS2021_00352", "BraTS2021_00840", "BraTS2021_01327", "BraTS2021_00167", "BraTS2021_00586", "BraTS2021_00338", "BraTS2021_01463", "BraTS2021_01107", "BraTS2021_00103", "BraTS2021_01025", "BraTS2021_00104", "BraTS2021_00270", "BraTS2021_01287", "BraTS2021_00117", "BraTS2021_00742", "BraTS2021_01505", "BraTS2021_01512", "BraTS2021_00781", "BraTS2021_01289", "BraTS2021_01357", "BraTS2021_01547", "BraTS2021_00692", "BraTS2021_01014", "BraTS2021_01665", "BraTS2021_00089", "BraTS2021_00454", "BraTS2021_01526", "BraTS2021_00096", "BraTS2021_00799", "BraTS2021_01169", "BraTS2021_01009", "BraTS2021_00090", "BraTS2021_00312", "BraTS2021_00429", "BraTS2021_00275", "BraTS2021_00222", "BraTS2021_00708", "BraTS2021_01627", "BraTS2021_01093", "BraTS2021_01605", "BraTS2021_01413", "BraTS2021_01164", "BraTS2021_01396", "BraTS2021_01129", "BraTS2021_01500", "BraTS2021_01572", "BraTS2021_00657", "BraTS2021_01219", "BraTS2021_01218", "BraTS2021_01432", "BraTS2021_00383", "BraTS2021_00570", "BraTS2021_00003", "BraTS2021_00555", "BraTS2021_00238", "BraTS2021_00094", "BraTS2021_01493", "BraTS2021_00582", "BraTS2021_01301", "BraTS2021_00306", "BraTS2021_00698", "BraTS2021_01469", "BraTS2021_00589", "BraTS2021_00240", "BraTS2021_00340", "BraTS2021_01285", "BraTS2021_01187", "BraTS2021_01364", "BraTS2021_00133", "BraTS2021_01408", "BraTS2021_00021", "BraTS2021_01272", "BraTS2021_00796", "BraTS2021_01083", "BraTS2021_00143", "BraTS2021_00285", "BraTS2021_01323", "BraTS2021_00218", "BraTS2021_00731", "BraTS2021_01363", "BraTS2021_01474", "BraTS2021_00539", "BraTS2021_01553", "BraTS2021_00405", "BraTS2021_01334", "BraTS2021_01466", "BraTS2021_00559", "BraTS2021_00729", "BraTS2021_00313", "BraTS2021_01110", "BraTS2021_01240", "BraTS2021_00171", "BraTS2021_00286", "BraTS2021_00151", "BraTS2021_01372", "BraTS2021_01257", "BraTS2021_01283", "BraTS2021_01141", "BraTS2021_01298", "BraTS2021_01628", "BraTS2021_00274", "BraTS2021_00556", "BraTS2021_01249", "BraTS2021_01470", "BraTS2021_00707", "BraTS2021_01384", "BraTS2021_00382", "BraTS2021_00618", "BraTS2021_01410", "BraTS2021_00818", "BraTS2021_00691", "BraTS2021_01468", "BraTS2021_01483", "BraTS2021_01092", "BraTS2021_00098", "BraTS2021_00194", "BraTS2021_00661", "BraTS2021_00165", "BraTS2021_00206", "BraTS2021_00020", "BraTS2021_01643", "BraTS2021_00668", "BraTS2021_00063", "BraTS2021_01108", "BraTS2021_00185", "BraTS2021_00651", "BraTS2021_00727", "BraTS2021_00347", "BraTS2021_01340", "BraTS2021_01475", "BraTS2021_01313", "BraTS2021_01180", "BraTS2021_01564", "BraTS2021_01054", "BraTS2021_00583", "BraTS2021_00611", "BraTS2021_01282", "BraTS2021_01496", "BraTS2021_01571", "BraTS2021_00485", "BraTS2021_01389", "BraTS2021_00371", "BraTS2021_00364", "BraTS2021_01506", "BraTS2021_00199", "BraTS2021_01411", "BraTS2021_00247", "BraTS2021_01619", "BraTS2021_00703", "BraTS2021_00160", "BraTS2021_00694", "BraTS2021_01045", "BraTS2021_01159", "BraTS2021_00321", "BraTS2021_00059", "BraTS2021_01416", "BraTS2021_01079", "BraTS2021_00002", "BraTS2021_01502", "BraTS2021_01520", "BraTS2021_00470", "BraTS2021_01181", "BraTS2021_01555", "BraTS2021_01646", "BraTS2021_01645", "BraTS2021_01027", "BraTS2021_00305", "BraTS2021_01455", "BraTS2021_00524", "BraTS2021_01441", "BraTS2021_00775", "BraTS2021_00009", "BraTS2021_01345", "BraTS2021_00456", "BraTS2021_01306", "BraTS2021_01017", "BraTS2021_01666", "BraTS2021_01636", "BraTS2021_01615", "BraTS2021_01515", "BraTS2021_00328", "BraTS2021_00032", "BraTS2021_00734", "BraTS2021_01499", "BraTS2021_01374", "BraTS2021_01197", "BraTS2021_01204", "BraTS2021_00184", "BraTS2021_01656", "BraTS2021_00466", "BraTS2021_00444", "BraTS2021_00314", "BraTS2021_00134", "BraTS2021_00263", "BraTS2021_01426", "BraTS2021_01459", "BraTS2021_00401", "BraTS2021_01588", "BraTS2021_01157", "BraTS2021_01233", "BraTS2021_01293", "BraTS2021_00774", "BraTS2021_01519", "BraTS2021_01647", "BraTS2021_00724", "BraTS2021_01661", "BraTS2021_00780", "BraTS2021_00412", "BraTS2021_01355", "BraTS2021_01226", "BraTS2021_01438", "BraTS2021_00613", "BraTS2021_01060", "BraTS2021_01577", "BraTS2021_00610", "BraTS2021_00191", "BraTS2021_00789", "BraTS2021_01237", "BraTS2021_01062", "BraTS2021_01655", "BraTS2021_00663", "BraTS2021_00156", "BraTS2021_01080", "BraTS2021_00631", "BraTS2021_01556", "BraTS2021_00795", "BraTS2021_01193", "BraTS2021_01362", "BraTS2021_00479", "BraTS2021_01663", "BraTS2021_01509", "BraTS2021_00504", "BraTS2021_00207", "BraTS2021_00787", "BraTS2021_00606", "BraTS2021_01143", "BraTS2021_01174", "BraTS2021_00478", "BraTS2021_01163", "BraTS2021_01344", "BraTS2021_01635", "BraTS2021_01629", "BraTS2021_00292", "BraTS2021_01651", "BraTS2021_00233", "BraTS2021_01008", "BraTS2021_00658", "BraTS2021_00131", "BraTS2021_00201", "BraTS2021_00561", "BraTS2021_01598", "BraTS2021_01156", "BraTS2021_01016", "BraTS2021_01546", "BraTS2021_00230", "BraTS2021_01191", "BraTS2021_00359", "BraTS2021_01514", "BraTS2021_01356", "BraTS2021_00124", "BraTS2021_00778", "BraTS2021_01120", "BraTS2021_01151", "BraTS2021_00283", "BraTS2021_01194", "BraTS2021_00269", "BraTS2021_01660", "BraTS2021_01105", "BraTS2021_00620", "BraTS2021_00451", "BraTS2021_01640", "BraTS2021_01112", "BraTS2021_01281", "BraTS2021_00464", "BraTS2021_00019", "BraTS2021_00816", "BraTS2021_01622", "BraTS2021_01335", "BraTS2021_00638", "BraTS2021_01275", "BraTS2021_01659", "BraTS2021_00579", "BraTS2021_01565", "BraTS2021_00084", "BraTS2021_01648", "BraTS2021_01450", "BraTS2021_00679", "BraTS2021_00628", "BraTS2021_00271", "BraTS2021_00095", "BraTS2021_00099", "BraTS2021_01280", "BraTS2021_01625", "BraTS2021_00251", "BraTS2021_01532", "BraTS2021_00296", "BraTS2021_00403", "BraTS2021_00646", "BraTS2021_00715", "BraTS2021_00807", "BraTS2021_00077", "BraTS2021_00320"]


len(a)

1126

In [27]:
b = ["BraTS2021_01417", "BraTS2021_01632", "BraTS2021_01664", "BraTS2021_01305", "BraTS2021_01339", "BraTS2021_01155", "BraTS2021_01642", "BraTS2021_00548", "BraTS2021_01271", "BraTS2021_01030", "BraTS2021_00148", "BraTS2021_00049", "BraTS2021_01464", "BraTS2021_00732", "BraTS2021_01533", "BraTS2021_00280", "BraTS2021_01070", "BraTS2021_01245", "BraTS2021_01002", "BraTS2021_00512", "BraTS2021_01224", "BraTS2021_01366", "BraTS2021_01220", "BraTS2021_00273", "BraTS2021_01551", "BraTS2021_00753", "BraTS2021_00370", "BraTS2021_01063", "BraTS2021_00053", "BraTS2021_01038", "BraTS2021_01284", "BraTS2021_01188", "BraTS2021_00652", "BraTS2021_01649", "BraTS2021_00390", "BraTS2021_01225", "BraTS2021_00072", "BraTS2021_01405", "BraTS2021_00501", "BraTS2021_01213", "BraTS2021_00253", "BraTS2021_01421", "BraTS2021_00716", "BraTS2021_01277", "BraTS2021_01461", "BraTS2021_00100", "BraTS2021_01623", "BraTS2021_00414", "BraTS2021_01630", "BraTS2021_00549", "BraTS2021_01203", "BraTS2021_01128", "BraTS2021_01140", "BraTS2021_00723", "BraTS2021_00419", "BraTS2021_01040", "BraTS2021_01147", "BraTS2021_00612", "BraTS2021_01476", "BraTS2021_00655", "BraTS2021_01511", "BraTS2021_01435", "BraTS2021_00139", "BraTS2021_01513", "BraTS2021_00605", "BraTS2021_01167", "BraTS2021_01015", "BraTS2021_01436", "BraTS2021_00339", "BraTS2021_01652", "BraTS2021_00728", "BraTS2021_01142", "BraTS2021_00030", "BraTS2021_01033", "BraTS2021_01307", "BraTS2021_01078", "BraTS2021_00085", "BraTS2021_00310", "BraTS2021_01422", "BraTS2021_01199", "BraTS2021_01137", "BraTS2021_01184", "BraTS2021_01440", "BraTS2021_00499", "BraTS2021_01358", "BraTS2021_01087", "BraTS2021_01210", "BraTS2021_00593", "BraTS2021_01448", "BraTS2021_00772", "BraTS2021_01488", "BraTS2021_00407", "BraTS2021_00373", "BraTS2021_01545", "BraTS2021_01603", "BraTS2021_00641", "BraTS2021_00266", "BraTS2021_01221", "BraTS2021_01311", "BraTS2021_00483", "BraTS2021_01048", "BraTS2021_01653", "BraTS2021_00758", "BraTS2021_00288", "BraTS2021_00025", "BraTS2021_00297", "BraTS2021_01148", "BraTS2021_00806", "BraTS2021_00101", "BraTS2021_00735", "BraTS2021_00087", "BraTS2021_00477", "BraTS2021_01597", "BraTS2021_01385", "BraTS2021_01391", "BraTS2021_01379", "BraTS2021_01332", "BraTS2021_01256", "BraTS2021_01491", "BraTS2021_00006", "BraTS2021_01003", "BraTS2021_01523", "BraTS2021_00801", "BraTS2021_01144", "BraTS2021_00518"]

print(len(b))

125
